# Pitch direction vs RF vertical position — paired comparison

Based on `tc_comparison_pairs.ipynb`, retaining the m14, m15 A+B/A/B, m19, m20, and pooled sections. Match the same units using the **existing HD Class 3 selection**, then compare **pitch tuning** with **RF vertical tuning**. No HD classes are recomputed here.

Pitch uses the already generated physical `pitch_direction` data: forward/level 0°, up positive, down negative. The reference-forward vector is calibrated from the confirmed level hp4 pose facing VS0 (`44.7956488331414°` in Motive world XY), transformed into the body frame, then rotated with each intrinsic XYZ pose. Pitch = `degrees(atan2(forward_z, hypot(forward_x, forward_y)))`. These pose outputs and existing HD caches are not regenerated by this notebook.

The freely moving pitch sessions follow the original paired notebook: **m14 260609_1 → RF_3; m15 260630_1 A+B → RF_3; m19 260827_9 → RF_2; m20 260921_9 → RF_2**. We compare firing-rate profiles for the same `(mouse, date, probe, unit_id)` across these paired recordings.

**RF vertical convention:** stored `yPositions` is the original `Square_PositionY`, positive downward. Plot/report **up-positive elevation = −yPositions**, matching the pitch sign. The original stimulus MATLAB code uses `pixYlim = screenHeight/2 + ceil(Square_PositionY/degPerPix) − halfSize`, then writes `imMtrx(j, i)` directly to the texture (remote `Kai/MatlabApps/stimulus-delivery/StimGenStimuli/ReceptiveFieldMapping_Fast_360degLED.m`, lines 237–278). The RF generator preserves that Y sign (`/mnt/ssd4.1/Matlab/Utils/RFmapping_core.m`, lines 155 and 203–206).

The default RF curve first normalizes each position by `occupancyTimeSec`, sums time bins in **[0, 0.2) s**, and sums over horizontal positions. The main peak is therefore the **1-D vertical-profile maximum**, consistent with paired tuning-curve analysis. The **elevation of the strongest 2-D bin** is a different measurement; both are saved, and `rf_peak_method` below selects which enters sorting/regression. Defaults retain saved 2-D RF detections, no smoothing, and ≥1 s pitch occupancy per native bin.

Both variables are bounded linear angles. Heatmap inputs are explicitly scaled per unit with `normalize_tc`, then shown in both native-peak sort orders; the scatter uses ordinary linear regression. There is **no 360° wrapping, circular smoothing, or circular correlation**. RF native Y grids differ between mice; only heatmap display is linearly resampled onto a common 1° grid, with no extrapolation. Peak positions and regression always use native bins.


In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib import pyplot as plt
from Utils.direction_comparison import normalize_tc
from Utils.plotting import LIGHT_PLOT_STYLE
from Utils.pitch_direction_comparison import (load_pitch_rf_pair, select_min_occupancy, detected_rf_units,
    select_pitch_rf_pair, resample_rf_profiles, plot_pitch_rf_pair, pitch_rf_regression, plot_pitch_rf_regression)

assert Path(sys.executable).samefile(Path.home() / '.virtualenvs/rfmapping/bin/python')
root = Path('/mnt/senzailab/Kai/#Recording')
cache_dir = Path('output/pitch_direction')
output_dir = Path('output/pitch_direction_rf_vertical')
figure_dir = output_dir / 'figures'
figure_dir.mkdir(parents=True, exist_ok=True)
plt.rcParams.update(LIGHT_PLOT_STYLE)
plot_options = dict(vmin=0, vmax=1, colorbar_label="Normalized response")
min_pitch_occupancy_s = 1.0
rf_window_s = (0.0, 0.2)
rf_type = '2d'  # Saved 2-D RF detections; None retains all RF units.
rf_peak_method = 'vertical_profile'  # Or 'max_bin_2d'; both values are saved.
rf_y_to_elevation_sign = -1  # Stored stimulus Y is downward; elevation is upward.
statistics_rows = []
source_rows = []

def load_recording(mouse, date, pitch_session, rf_session, probe):
    rf_source = root / mouse / str(date) / f'{date}_{rf_session}' / 'data/rfmapping/good/-100_400_1ms' / f'Probe{probe}' / f'regular_unitsSpikeCounts_{date}_{rf_session}.rfmap'
    pitch, native_rf, native_peaks = load_pitch_rf_pair(
        cache_dir, rf_source, mouse=mouse, date=date, pitch_session=pitch_session, probe=probe,
        y_to_elevation_sign=rf_y_to_elevation_sign, window=rf_window_s,
    )
    pitch = select_min_occupancy(pitch, min_occupancy_s=min_pitch_occupancy_s)
    detected_units = detected_rf_units(rf_source) if rf_type == '2d' else None
    pitch, native_rf, peaks, selection = select_pitch_rf_pair(
        pitch, native_rf, native_peaks, rf_unit_ids=detected_units, rf_peak_method=rf_peak_method,
    )
    selection.to_csv(output_dir / f'{mouse}_{date}_Probe{probe}_unit_selection.csv')
    rf = resample_rf_profiles(native_rf)
    source_rows.append(dict(mouse=mouse, date=date, probe=probe, pitch_session=pitch_session,
                            rf_session=rf_session, units=len(peaks), rf_source=str(rf_source)))
    return pitch, rf, peaks

def show_save(figures, prefix):
    for i, figure in enumerate(figures):
        figure.savefig(figure_dir / f'{prefix}_{i}.png', dpi=150, facecolor='white', transparent=False)
        display(figure)
        plt.close(figure)


## m14 260609 Probe A: pitch 1 / RF 3

Matched units; native pitch and RF elevation peaks; both signs positive upward.


In [2]:
pitch_m14, rf_m14, peaks_m14 = load_recording('m14', '260609', 1, 3, 'A')
print('Eligible Class 3 / RF shared units:', len(peaks_m14))


Eligible Class 3 / RF shared units: 13


In [ ]:
figures = plot_pitch_rf_pair(normalize_tc(pitch_m14), normalize_tc(rf_m14), order=peaks_m14.sort_values('pitch_peak_deg', kind='stable').index.tolist(), sort_label='pitch', label="m14 260609 Probe A: pitch 1 / RF 3", **plot_options)
show_save(figures, 'm14_sorted_pitch')
figures = plot_pitch_rf_pair(normalize_tc(pitch_m14), normalize_tc(rf_m14), order=peaks_m14.sort_values('rf_peak_deg', kind='stable').index.tolist(), sort_label='rf', label="m14 260609 Probe A: pitch 1 / RF 3", **plot_options)
show_save(figures, 'm14_sorted_rf')


In [ ]:
statistics_m14 = pitch_rf_regression(peaks_m14)
figure = plot_pitch_rf_regression(peaks_m14, statistics_m14, label="m14 260609 Probe A: pitch 1 / RF 3", rf_peak_method=rf_peak_method)
show_save([figure], 'm14_regression')
statistics_rows.append({'group': 'm14', 'rf_peak_method': rf_peak_method, **statistics_m14})
peaks_m14.to_csv(output_dir / 'm14_preferred_angles.csv')
display(pd.DataFrame([statistics_m14]))


## m15 260630 Probe A+B: pitch 1 / RF 3

Matched units; native pitch and RF elevation peaks; both signs positive upward.


In [5]:
pitch_m15_A, rf_m15_A, peaks_m15_A = load_recording('m15', '260630', 1, 3, 'A')
pitch_m15_B, rf_m15_B, peaks_m15_B = load_recording('m15', '260630', 1, 3, 'B')
pitch_m15 = pd.concat([pitch_m15_A, pitch_m15_B])
rf_m15 = pd.concat([rf_m15_A, rf_m15_B])
peaks_m15 = pd.concat([peaks_m15_A, peaks_m15_B])
print('Eligible Class 3 / RF shared units:', len(peaks_m15))


Eligible Class 3 / RF shared units: 50


In [ ]:
figures = plot_pitch_rf_pair(normalize_tc(pitch_m15), normalize_tc(rf_m15), order=peaks_m15.sort_values('pitch_peak_deg', kind='stable').index.tolist(), sort_label='pitch', label="m15 260630 Probe A+B: pitch 1 / RF 3", **plot_options)
show_save(figures, 'm15_sorted_pitch')
figures = plot_pitch_rf_pair(normalize_tc(pitch_m15), normalize_tc(rf_m15), order=peaks_m15.sort_values('rf_peak_deg', kind='stable').index.tolist(), sort_label='rf', label="m15 260630 Probe A+B: pitch 1 / RF 3", **plot_options)
show_save(figures, 'm15_sorted_rf')


In [ ]:
statistics_m15 = pitch_rf_regression(peaks_m15)
figure = plot_pitch_rf_regression(peaks_m15, statistics_m15, label="m15 260630 Probe A+B: pitch 1 / RF 3", rf_peak_method=rf_peak_method)
show_save([figure], 'm15_regression')
statistics_rows.append({'group': 'm15', 'rf_peak_method': rf_peak_method, **statistics_m15})
peaks_m15.to_csv(output_dir / 'm15_preferred_angles.csv')
display(pd.DataFrame([statistics_m15]))


## m15 260630 Probe A only: pitch 1 / RF 3

Matched units; native pitch and RF elevation peaks; both signs positive upward.


In [8]:
# Same sessions, native peaks, Class 3 selection, and occupancy limit as A+B above.
print('Eligible Class 3 / RF shared units:', len(peaks_m15_A))


Eligible Class 3 / RF shared units: 25


In [ ]:
figures = plot_pitch_rf_pair(normalize_tc(pitch_m15_A), normalize_tc(rf_m15_A), order=peaks_m15_A.sort_values('pitch_peak_deg', kind='stable').index.tolist(), sort_label='pitch', label="m15 260630 Probe A only: pitch 1 / RF 3", **plot_options)
show_save(figures, 'm15_A_sorted_pitch')
figures = plot_pitch_rf_pair(normalize_tc(pitch_m15_A), normalize_tc(rf_m15_A), order=peaks_m15_A.sort_values('rf_peak_deg', kind='stable').index.tolist(), sort_label='rf', label="m15 260630 Probe A only: pitch 1 / RF 3", **plot_options)
show_save(figures, 'm15_A_sorted_rf')


In [ ]:
statistics_m15_A = pitch_rf_regression(peaks_m15_A)
figure = plot_pitch_rf_regression(peaks_m15_A, statistics_m15_A, label="m15 260630 Probe A only: pitch 1 / RF 3", rf_peak_method=rf_peak_method)
show_save([figure], 'm15_A_regression')
statistics_rows.append({'group': 'm15_A', 'rf_peak_method': rf_peak_method, **statistics_m15_A})
peaks_m15_A.to_csv(output_dir / 'm15_A_preferred_angles.csv')
display(pd.DataFrame([statistics_m15_A]))


## m15 260630 Probe B only: pitch 1 / RF 3

Matched units; native pitch and RF elevation peaks; both signs positive upward.


In [11]:
# Same sessions, native peaks, Class 3 selection, and occupancy limit as A+B above.
print('Eligible Class 3 / RF shared units:', len(peaks_m15_B))


Eligible Class 3 / RF shared units: 25


In [ ]:
figures = plot_pitch_rf_pair(normalize_tc(pitch_m15_B), normalize_tc(rf_m15_B), order=peaks_m15_B.sort_values('pitch_peak_deg', kind='stable').index.tolist(), sort_label='pitch', label="m15 260630 Probe B only: pitch 1 / RF 3", **plot_options)
show_save(figures, 'm15_B_sorted_pitch')
figures = plot_pitch_rf_pair(normalize_tc(pitch_m15_B), normalize_tc(rf_m15_B), order=peaks_m15_B.sort_values('rf_peak_deg', kind='stable').index.tolist(), sort_label='rf', label="m15 260630 Probe B only: pitch 1 / RF 3", **plot_options)
show_save(figures, 'm15_B_sorted_rf')


In [ ]:
statistics_m15_B = pitch_rf_regression(peaks_m15_B)
figure = plot_pitch_rf_regression(peaks_m15_B, statistics_m15_B, label="m15 260630 Probe B only: pitch 1 / RF 3", rf_peak_method=rf_peak_method)
show_save([figure], 'm15_B_regression')
statistics_rows.append({'group': 'm15_B', 'rf_peak_method': rf_peak_method, **statistics_m15_B})
peaks_m15_B.to_csv(output_dir / 'm15_B_preferred_angles.csv')
display(pd.DataFrame([statistics_m15_B]))


## m19 260827 Probe A: pitch 9 / RF 2

Matched units; native pitch and RF elevation peaks; both signs positive upward.


In [14]:
pitch_m19, rf_m19, peaks_m19 = load_recording('m19', '260827', 9, 2, 'A')
print('Eligible Class 3 / RF shared units:', len(peaks_m19))


Eligible Class 3 / RF shared units: 16


In [ ]:
figures = plot_pitch_rf_pair(normalize_tc(pitch_m19), normalize_tc(rf_m19), order=peaks_m19.sort_values('pitch_peak_deg', kind='stable').index.tolist(), sort_label='pitch', label="m19 260827 Probe A: pitch 9 / RF 2", **plot_options)
show_save(figures, 'm19_sorted_pitch')
figures = plot_pitch_rf_pair(normalize_tc(pitch_m19), normalize_tc(rf_m19), order=peaks_m19.sort_values('rf_peak_deg', kind='stable').index.tolist(), sort_label='rf', label="m19 260827 Probe A: pitch 9 / RF 2", **plot_options)
show_save(figures, 'm19_sorted_rf')


In [ ]:
statistics_m19 = pitch_rf_regression(peaks_m19)
figure = plot_pitch_rf_regression(peaks_m19, statistics_m19, label="m19 260827 Probe A: pitch 9 / RF 2", rf_peak_method=rf_peak_method)
show_save([figure], 'm19_regression')
statistics_rows.append({'group': 'm19', 'rf_peak_method': rf_peak_method, **statistics_m19})
peaks_m19.to_csv(output_dir / 'm19_preferred_angles.csv')
display(pd.DataFrame([statistics_m19]))


## m20 260921 Probe A: pitch 9 / RF 2

Matched units; native pitch and RF elevation peaks; both signs positive upward.


In [17]:
pitch_m20, rf_m20, peaks_m20 = load_recording('m20', '260921', 9, 2, 'A')
print('Eligible Class 3 / RF shared units:', len(peaks_m20))


Eligible Class 3 / RF shared units: 14


In [ ]:
figures = plot_pitch_rf_pair(normalize_tc(pitch_m20), normalize_tc(rf_m20), order=peaks_m20.sort_values('pitch_peak_deg', kind='stable').index.tolist(), sort_label='pitch', label="m20 260921 Probe A: pitch 9 / RF 2", **plot_options)
show_save(figures, 'm20_sorted_pitch')
figures = plot_pitch_rf_pair(normalize_tc(pitch_m20), normalize_tc(rf_m20), order=peaks_m20.sort_values('rf_peak_deg', kind='stable').index.tolist(), sort_label='rf', label="m20 260921 Probe A: pitch 9 / RF 2", **plot_options)
show_save(figures, 'm20_sorted_rf')


In [ ]:
statistics_m20 = pitch_rf_regression(peaks_m20)
figure = plot_pitch_rf_regression(peaks_m20, statistics_m20, label="m20 260921 Probe A: pitch 9 / RF 2", rf_peak_method=rf_peak_method)
show_save([figure], 'm20_regression')
statistics_rows.append({'group': 'm20', 'rf_peak_method': rf_peak_method, **statistics_m20})
peaks_m20.to_csv(output_dir / 'm20_preferred_angles.csv')
display(pd.DataFrame([statistics_m20]))


## All four mice: m14 + m15 + m19 + m20

Matched units; native pitch and RF elevation peaks; both signs positive upward.


In [20]:
pitch_all = pd.concat([pitch_m14, pitch_m15, pitch_m19, pitch_m20])
rf_all = pd.concat([rf_m14, rf_m15, rf_m19, rf_m20])
peaks_all = pd.concat([peaks_m14, peaks_m15, peaks_m19, peaks_m20])
assert pitch_all.index.is_unique and pitch_all.index.equals(rf_all.index)
print('Eligible Class 3 / RF shared units:', len(peaks_all))


Eligible Class 3 / RF shared units: 93


In [ ]:
figures = plot_pitch_rf_pair(normalize_tc(pitch_all), normalize_tc(rf_all), order=peaks_all.sort_values('pitch_peak_deg', kind='stable').index.tolist(), sort_label='pitch', label="All four mice: m14 + m15 + m19 + m20", **plot_options)
show_save(figures, 'all_sorted_pitch')
figures = plot_pitch_rf_pair(normalize_tc(pitch_all), normalize_tc(rf_all), order=peaks_all.sort_values('rf_peak_deg', kind='stable').index.tolist(), sort_label='rf', label="All four mice: m14 + m15 + m19 + m20", **plot_options)
show_save(figures, 'all_sorted_rf')


In [ ]:
statistics_all = pitch_rf_regression(peaks_all)
figure = plot_pitch_rf_regression(peaks_all, statistics_all, label="All four mice: m14 + m15 + m19 + m20", rf_peak_method=rf_peak_method)
show_save([figure], 'all_regression')
statistics_rows.append({'group': 'all', 'rf_peak_method': rf_peak_method, **statistics_all})
peaks_all.to_csv(output_dir / 'all_preferred_angles.csv')
display(pd.DataFrame([statistics_all]))


## Pitch–RF vertical regression results

OLS fits `RF elevation = intercept + slope × preferred pitch`; `R²` and two-sided slope-test `p` are reported. These are descriptive unit-level comparisons, not independent animal-level inference. A+B, A, B, and pooled sections share units; their unadjusted p-values are not independent replications. Marginal pitch tuning can reflect behavioral covariance. Native peak ties are resolved by the first ascending elevation/pitch bin.

`rf_vertical_profile_peak_deg` is the peak after summing over horizontal RF positions. `rf_2d_max_elevation_deg` is the vertical location of the single strongest 2-D response bin; it can differ, and changing `rf_peak_method` does not change the matched cohort or displayed vertical profiles. All output angles are up-positive degrees.


In [23]:
statistics_table = pd.DataFrame(statistics_rows)
statistics_table.to_csv(output_dir / 'comparison_statistics.csv', index=False)
source_table = pd.DataFrame(source_rows)
source_table.to_csv(output_dir / 'source_sessions.csv', index=False)
(output_dir / 'analysis_parameters.json').write_text(json.dumps(dict(
    rf_window_s=rf_window_s, rf_type=rf_type, rf_peak_method=rf_peak_method,
    min_pitch_occupancy_s=min_pitch_occupancy_s, rf_y_to_elevation_sign=rf_y_to_elevation_sign,
    rf_vertical_definition='sum_x(sum_[0,0.2)s(spike_counts / occupancyTimeSec)); native peak',
    rf_vertical_sign_source='/mnt/senzailab/Kai/MatlabApps/stimulus-delivery/StimGenStimuli/ReceptiveFieldMapping_Fast_360degLED.m:237-278',
    pitch_class_source='Existing HD Class 3 cache; classification unchanged',
    rf_heatmap_display_grid_deg='-45:1:45, linear interpolation only; native peaks retained',
), indent=2))
display(source_table)
display(statistics_table)


,mouse,date,probe,pitch_session,rf_session,units,rf_source
0,m14,260609,A,1,3,13,/mnt/senzailab/Kai/#Recording/m14/260609/26060...
1,m15,260630,A,1,3,25,/mnt/senzailab/Kai/#Recording/m15/260630/26063...
2,m15,260630,B,1,3,25,/mnt/senzailab/Kai/#Recording/m15/260630/26063...
3,m19,260827,A,9,2,16,/mnt/senzailab/Kai/#Recording/m19/260827/26082...
4,m20,260921,A,9,2,14,/mnt/senzailab/Kai/#Recording/m20/260921/26092...


,group,rf_peak_method,n,slope,intercept,r,r_squared,p
0,m14,vertical_profile,13,0.027707,1.054987,0.051111,0.002612,0.868297
1,m15,vertical_profile,50,0.112446,16.643197,0.097820,0.009569,0.499155
2,m15_A,vertical_profile,25,0.068221,15.124443,0.060346,0.003642,0.774463
3,m15_B,vertical_profile,25,0.188406,19.304348,0.157912,0.024936,0.450919
4,m19,vertical_profile,16,0.181034,9.206897,0.336498,0.113231,0.202533
5,m20,vertical_profile,14,-0.048212,8.162846,-0.094565,0.008942,0.747785
6,all,vertical_profile,93,0.068069,11.785868,0.114527,0.013117,0.274339
